# 02 - 自訂 Workflow：建立多步驟 LLM 工作流

本筆記本示範如何使用本框架的 `BaseWorkflow` 與 LangGraph 建立一個自訂的多步驟工作流程。

## 簡介

本框架以 [LangGraph](https://github.com/langchain-ai/langgraph) 的 `StateGraph` 為核心，提供 `BaseWorkflow` 這個高階 wrapper，讓你能以 **builder pattern（鏈式呼叫）** 快速組裝多步驟工作流程：

```
BaseWorkflow(name, state_schema)
    .add_node(name, func)   # 新增節點
    .add_edge(src, tgt)     # 新增有向邊
    .set_entry(name)        # 設定入口節點
    .set_finish(name)       # 設定終止節點
    .compile()              # 編譯 graph
    .run(state)             # 同步執行
```

**核心概念：**
- **State**：貫穿整個 workflow 的共享資料結構（TypedDict）。
- **Node**：每個處理步驟，是一個接受 state dict 並回傳部分更新的普通函式。
- **Edge**：定義節點之間的執行順序。
- **Tools**：框架提供 `parse_json`、`validate_output` 等實用工具，方便處理 LLM 輸出。

In [ ]:
import sys
import os

# 確保 project root 在 Python path 中
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from app.utils.config import init_config
from app.workflow import BaseWorkflow, WorkflowState, create_workflow_state, WorkflowError
from app.workflow.tools import parse_json, validate_output, get_structured_output
from pydantic import BaseModel

# 初始化設定（dev 環境）
cfg = init_config(env="dev")
print("設定載入完成，環境：", cfg.env)

## 定義 Workflow State

`WorkflowState` 是一個 `TypedDict`，包含以下欄位：

| 欄位 | 型別 | 說明 |
|---|---|---|
| `messages` | `list` | 對話訊息（使用 `add_messages` reducer，支援自動合併） |
| `metadata` | `dict` | 任意 key-value 中繼資料 |
| `current_step` | `str` | 目前執行的步驟名稱 |
| `results` | `dict` | 各節點的輸出結果 |
| `retry_count` | `int` | 重試次數 |
| `error` | `str \| None` | 錯誤訊息（無錯誤時為 `None`） |

使用 `create_workflow_state()` factory function 建立初始 state，所有參數皆有預設值。

In [ ]:
# 使用 factory function 建立初始 state
initial_state = create_workflow_state(
    messages=[
        {"role": "user", "content": "請分析以下文字：LLM 是一種大型語言模型，能夠理解並生成自然語言。"}
    ],
    metadata={"task": "text_analysis", "language": "zh-TW"},
    current_step="init",
)

print("初始 State 結構：")
for key, value in initial_state.items():
    print(f"  {key}: {value!r}")

## 建立 Node 函式

每個 **Node** 就是一個普通的 Python 函式，它：

1. **接受**完整的 `state` dict 作為唯一參數。
2. **回傳** 一個只包含需要更新欄位的 dict（部分更新，無需回傳整個 state）。

LangGraph 會自動將每個 node 的回傳值合併回 state，`messages` 欄位會用 `add_messages` reducer 做串接，其他欄位則直接覆蓋。

以下定義三個 node：
- **`prepare`**：擷取輸入內容，並記錄在 `results` 中。
- **`process`**：對輸入文字做基本分析（字數、關鍵詞）。
- **`output`**：整理最終輸出，附加 assistant 回覆訊息。

In [ ]:
def prepare(state: dict) -> dict:
    """Node 1：準備階段 — 擷取使用者輸入，初始化處理參數。"""
    messages = state.get("messages", [])
    user_text = ""
    for msg in reversed(messages):
        if msg.get("role") == "user":
            user_text = msg["content"]
            break

    print(f"[prepare] 擷取輸入文字（前 40 字）：{user_text[:40]}...")

    return {
        "current_step": "prepare",
        "results": {
            **state.get("results", {}),
            "raw_input": user_text,
        },
    }


def process(state: dict) -> dict:
    """Node 2：處理階段 — 對輸入文字做基本文字分析。"""
    raw_input: str = state.get("results", {}).get("raw_input", "")

    # 基本文字分析（不呼叫 LLM，僅示範 node 邏輯）
    char_count = len(raw_input)
    word_count = len(raw_input.split())
    # 簡單關鍵詞擷取：長度 >= 2 的中文詞彙片段
    keywords = list({w for w in raw_input.replace("，", " ").replace("。", " ").split() if len(w) >= 2})

    analysis = {
        "char_count": char_count,
        "word_count": word_count,
        "keywords": keywords[:5],  # 最多取前 5 個
        "language": state.get("metadata", {}).get("language", "unknown"),
    }

    print(f"[process] 分析完成：字元數={char_count}, 關鍵詞={keywords[:5]}")

    return {
        "current_step": "process",
        "results": {
            **state.get("results", {}),
            "analysis": analysis,
        },
    }


def output(state: dict) -> dict:
    """Node 3：輸出階段 — 整理分析結果並產生回覆訊息。"""
    analysis = state.get("results", {}).get("analysis", {})

    summary = (
        f"分析結果摘要：\n"
        f"  - 字元數：{analysis.get('char_count', 'N/A')}\n"
        f"  - 詞彙數：{analysis.get('word_count', 'N/A')}\n"
        f"  - 關鍵詞：{', '.join(analysis.get('keywords', []))}\n"
        f"  - 語言：{analysis.get('language', 'N/A')}"
    )

    print(f"[output] 產生輸出摘要")

    return {
        "current_step": "output",
        "messages": [{"role": "assistant", "content": summary}],
        "results": {
            **state.get("results", {}),
            "summary": summary,
        },
    }


print("三個 node 函式定義完成：prepare -> process -> output")

## 組裝 Workflow

使用 `BaseWorkflow` 的 builder pattern 將三個 node 組裝成一個完整的 workflow graph：

1. `add_node()` — 登錄每個節點（名稱 + 函式）。
2. `add_edge()` — 定義節點間的執行順序。
3. `set_entry()` — 設定 graph 的入口節點。
4. `set_finish()` — 設定終止節點，自動連接到 LangGraph 的 `END`。
5. `compile()` — 編譯並鎖定 graph（編譯後無法再修改）。

Graph 結構：`prepare` → `process` → `output` → `END`

In [ ]:
# 使用 builder pattern 組裝並編譯 workflow
workflow = (
    BaseWorkflow("text_analysis_workflow", WorkflowState)
    .add_node("prepare", prepare)
    .add_node("process", process)
    .add_node("output", output)
    .add_edge("prepare", "process")
    .add_edge("process", "output")
    .set_entry("prepare")
    .set_finish("output")
    .compile()
)

print(f"Workflow '{workflow.name}' 組裝完成")
print(f"入口節點：{workflow._entry_node}")
print(f"終止節點：{workflow._finish_nodes}")

## 執行 Workflow

呼叫 `workflow.run(state)` 同步執行 workflow。

- 若 MLflow 已啟用，`run()` 會自動建立 span 追蹤執行過程（graceful degradation：MLflow 不可用時仍可正常執行）。
- 回傳值是執行完畢後的最終 `state` dict。
- 若執行失敗，會拋出 `WorkflowError`。

In [ ]:
# 建立初始 state 並執行 workflow
initial_state = create_workflow_state(
    messages=[
        {"role": "user", "content": "請分析以下文字：LLM 是一種大型語言模型，能夠理解並生成自然語言。"}
    ],
    metadata={"task": "text_analysis", "language": "zh-TW"},
)

try:
    final_state = workflow.run(initial_state)
    print("\n=== Workflow 執行完成 ===")
    print(f"最終步驟：{final_state['current_step']}")
    print(f"錯誤：{final_state['error']}")
    print(f"\n訊息記錄（共 {len(final_state['messages'])} 則）：")
    for msg in final_state["messages"]:
        print(f"  [{msg['role']}] {msg['content'][:80]}")
    print(f"\n結果摘要：")
    print(final_state["results"].get("summary", "(無)"))
except WorkflowError as e:
    print(f"Workflow 執行失敗：{e}")

## 使用 Structured Output 工具

框架提供兩個實用工具，方便處理 LLM 的非結構化輸出：

### `parse_json(text)`
從 LLM 回傳的文字（可能包含 markdown code fence、trailing comma、單引號等常見問題）中提取並修復 JSON。

### `validate_output(data, Schema)`
將解析後的 dict 透過 Pydantic model 進行驗證，確保輸出符合預期結構；驗證失敗時拋出 `ValidationError`。

### `get_structured_output(client, messages, Schema)`
整合 LLM 呼叫、JSON 解析與驗證，支援自動重試（當解析或驗證失敗時，會將錯誤回饋給 LLM 並重新請求）。

以下示範 `parse_json` 與 `validate_output` 的基本用法：

In [ ]:
from typing import Optional

# 1. 定義 Pydantic 輸出 schema
class TextAnalysisResult(BaseModel):
    char_count: int
    word_count: int
    keywords: list[str]
    language: str
    summary: Optional[str] = None

print("=== 示範 parse_json ===")

# 模擬 LLM 回傳的非規範 JSON（含 markdown fence 與 trailing comma）
llm_raw_output = '''
以下是分析結果：
```json
{
  "char_count": 28,
  "word_count": 3,
  "keywords": ["LLM", "大型語言模型", "自然語言",],
  "language": "zh-TW",
  "summary": "此文字介紹 LLM 的基本概念",
}
```
'''

# parse_json 自動處理 markdown fence 與 trailing comma
parsed_data = parse_json(llm_raw_output)
print(f"parse_json 解析結果（型別：{type(parsed_data).__name__}）：")
for k, v in parsed_data.items():
    print(f"  {k}: {v!r}")

print("\n=== 示範 validate_output ===")

# validate_output 將 dict 驗證並轉換為 Pydantic model instance
validated = validate_output(parsed_data, TextAnalysisResult)
print(f"validate_output 結果（型別：{type(validated).__name__}）：")
print(f"  char_count : {validated.char_count}")
print(f"  word_count : {validated.word_count}")
print(f"  keywords   : {validated.keywords}")
print(f"  language   : {validated.language}")
print(f"  summary    : {validated.summary}")

print("\n=== 示範驗證失敗的處理 ===")
from pydantic import ValidationError

bad_data = {"char_count": "not_a_number", "word_count": 3, "keywords": [], "language": "zh-TW"}
try:
    validate_output(bad_data, TextAnalysisResult)
except ValidationError as e:
    print(f"驗證失敗（預期行為）：{e.error_count()} 個錯誤")
    for err in e.errors():
        print(f"  - {err['loc']}: {err['msg']}")

## 小結與後續步驟

### 本筆記本涵蓋的內容

- **`WorkflowState`**：多步驟 workflow 的共享狀態結構，使用 `create_workflow_state()` 建立初始值。
- **Node 函式**：每個節點是一個接受 `state` dict 並回傳部分更新的普通函式，易於測試與組合。
- **`BaseWorkflow` builder pattern**：透過 `add_node` / `add_edge` / `set_entry` / `set_finish` / `compile` 組裝 LangGraph 的 `StateGraph`。
- **`parse_json`**：自動提取並修復 LLM 輸出中的非規範 JSON。
- **`validate_output`**：以 Pydantic model 驗證解析後的資料，確保輸出結構正確。

### 後續可探索的功能

1. **條件邊（Conditional Edge）**：使用 `add_conditional_edge()` 根據 state 動態決定下一個節點，實現分支邏輯。
2. **真實 LLM 呼叫**：在 node 函式中使用 `app.api.llm_client` 進行實際的 LLM API 呼叫。
3. **`get_structured_output`**：整合 LLM 呼叫、JSON 解析與 Pydantic 驗證，並自動重試失敗的輸出提取。
4. **非同步執行**：使用 `workflow.arun()` 與 `aget_structured_output()` 進行非同步執行，提升吞吐量。
5. **MLflow 追蹤**：`workflow.run()` 已內建 MLflow span 追蹤，可在 MLflow UI 查看執行記錄與 latency。
6. **自訂 State Schema**：繼承或自訂 `TypedDict` 以滿足特定 workflow 的狀態需求。

請參考 `app/workflow/` 目錄的原始碼與後續範例筆記本以深入了解各功能的實作細節。